In [4]:
import os
import sys
from pyspark.sql import SparkSession

os.environ["JAVA_HOME"]             = r"C:\Program Files\Java\jdk-17"
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

JDBC_JAR = os.path.join(os.getcwd(), "mssql-jdbc-13.4.0.jre11.jar")

spark = (
    SparkSession.builder
    .appName("AirQualityAnalysis")
    .master("local[*]")
    .config("spark.jars",                    JDBC_JAR)
    .config("spark.driver.extraClassPath",   JDBC_JAR)
    .config("spark.executor.extraClassPath", JDBC_JAR)
    .config("spark.driver.memory",           "2g")
    .config("spark.sql.shuffle.partitions",  "4")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark ready:", spark.version)

✅ Spark ready: 4.1.1


In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

CSV_PATH = "live_input/air_quality_combined.csv"

raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("encoding", "UTF-8")
    .option("nullValue", "")
    .csv(CSV_PATH)
)

print("✅ Raw rows:", raw_df.count())

✅ Raw rows: 1418


In [6]:
def clean(df):
    # Step 1 — cast numerics
    for col in ["aqi", "pm25", "pm10", "co", "no2", "so2", "o3"]:
        df = df.withColumn(col, F.col(col).cast(DoubleType()))
    df = df.withColumn("latitude",  F.col("latitude").cast(DoubleType()))
    df = df.withColumn("longitude", F.col("longitude").cast(DoubleType()))

    # Step 2 — parse datetime
    df = df.withColumn(
        "datetime",
        F.to_timestamp("datetime", "yyyy-MM-dd'T'HH:mm")
    )

    # Step 3 — remove nulls
    df = df.filter(F.col("pm25").isNotNull())
    df = df.filter(F.col("datetime").isNotNull())

    # Step 4 — remove negatives
    df = df.filter(
        (F.col("pm25") >= 0) &
        (F.col("pm10") >= 0) &
        (F.col("aqi")  >= 0)
    )

    # Step 5 — add time columns
    df = df.withColumn("date",  F.to_date("datetime"))
    df = df.withColumn("hour",  F.hour("datetime"))
    df = df.withColumn("month", F.month("datetime"))

    # Step 6 — add category
    df = df.withColumn(
        "air_quality_category",
        F.when(F.col("pm25") <= 12.0,  "Good")
         .when(F.col("pm25") <= 35.4,  "Moderate")
         .when(F.col("pm25") <= 55.4,  "Unhealthy for Sensitive Groups")
         .when(F.col("pm25") <= 150.4, "Unhealthy")
         .when(F.col("pm25") <= 250.4, "Very Unhealthy")
         .otherwise("Hazardous")
    )

    # Step 7 — NO dropDuplicates (API data has same city+datetime = valid history)
    return df

# Test
spark.catalog.clearCache()
raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("nullValue", "")
    .option("encoding", "UTF-8")
    .csv("live_input/air_quality_combined.csv")
)

clean_df = clean(raw_df)
print("Raw rows:  ", raw_df.count())
print("Clean rows:", clean_df.count())  # should be ~1003 now

Raw rows:   1418
Clean rows: 1418


In [7]:
avg_pm25 = (
    clean_df.groupBy("city")
    .agg(
        F.round(F.avg("pm25"), 2).alias("avg_pm25"),
        F.round(F.max("pm25"), 2).alias("max_pm25"),
        F.count("*").alias("total_records")
    )
    .orderBy(F.desc("avg_pm25"))
)

pollutants = (
    clean_df.groupBy("city")
    .agg(
        F.round(F.avg("pm25"), 2).alias("avg_pm25"),
        F.round(F.avg("pm10"), 2).alias("avg_pm10"),
        F.round(F.avg("co"),   2).alias("avg_co"),
        F.round(F.avg("no2"),  2).alias("avg_no2"),
        F.round(F.avg("so2"),  2).alias("avg_so2"),
        F.round(F.avg("o3"),   2).alias("avg_o3"),
    )
    .orderBy(F.desc("avg_pm25"))
)

dangerous = (
    clean_df.filter(
        F.col("air_quality_category").isin(
            "Unhealthy", "Very Unhealthy", "Hazardous"
        )
    )
    .groupBy("city", "air_quality_category")
    .agg(F.count("*").alias("dangerous_count"))
    .orderBy(F.desc("dangerous_count"))
)

print("✅ Aggregations ready")

✅ Aggregations ready


In [8]:
import urllib
from sqlalchemy import create_engine, text

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=(local)\\MSSQLSERVER01;"
    "DATABASE=AirQualityDB;"
    "UID=airuser;"
    "PWD=Air@12345;"
    "TrustServerCertificate=yes;"
)

params = urllib.parse.quote_plus(conn_str)
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

def save_to_sql(spark_df, table_name, if_exists="append"):
    print(f"\n  Saving → {table_name}")
    try:
        pdf = spark_df.toPandas()
        print(f"    Rows: {len(pdf)}")
        tbl = table_name.replace("dbo.", "")
        pdf.to_sql(
            tbl,
            engine,
            schema="dbo",
            if_exists=if_exists,
            index=False,
            chunksize=100,
            method=None
        )
        print(f"    ✅ Done → {table_name}")   # ← was missing
    except Exception as e:
        import traceback
        print(f"    ❌ FAILED → {table_name}")
        traceback.print_exc()

print("Saving all tables...")
save_to_sql(clean_df,   "dbo.CleanedAirQuality",  if_exists="append")
save_to_sql(avg_pm25,   "dbo.CitySummary",         if_exists="replace")
save_to_sql(pollutants, "dbo.PollutantAvg",        if_exists="replace")
save_to_sql(dangerous,  "dbo.DangerousAirQuality", if_exists="replace")
print("\n✅ All tables saved!")

Saving all tables...

  Saving → dbo.CleanedAirQuality
    Rows: 1418
    ✅ Done → dbo.CleanedAirQuality

  Saving → dbo.CitySummary
    Rows: 5
    ✅ Done → dbo.CitySummary

  Saving → dbo.PollutantAvg
    Rows: 5
    ✅ Done → dbo.PollutantAvg

  Saving → dbo.DangerousAirQuality
    Rows: 0
    ✅ Done → dbo.DangerousAirQuality

✅ All tables saved!


In [ ]:
import time
from datetime import datetime

CSV_PATH = "live_input/air_quality_combined.csv"
INTERVAL = 60

print("=" * 50)
print("Batch loop started — Interrupt kernel to stop")
print("=" * 50)

batch = 0

while True:
    batch += 1
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\n[Batch {batch}] {now}")

    try:
        # ── Read fresh CSV ────────────────────────
        spark.catalog.clearCache()
        raw_df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "false")
            .option("nullValue", "")
            .option("encoding", "UTF-8")
            .csv(CSV_PATH)
        )

        # ── Clean ─────────────────────────────────
        clean_df = clean(raw_df)
        print(f"  Raw rows:   {raw_df.count()}")
        print(f"  Clean rows: {clean_df.count()}")

        # ── Aggregations ──────────────────────────
        avg_pm25 = clean_df.groupBy("city").agg(
            F.round(F.avg("pm25"), 2).alias("avg_pm25"),
            F.round(F.max("pm25"), 2).alias("max_pm25"),
            F.count("*").alias("total_records")
        ).orderBy(F.desc("avg_pm25"))

        pollutants = clean_df.groupBy("city").agg(
            F.round(F.avg("pm25"), 2).alias("avg_pm25"),
            F.round(F.avg("pm10"), 2).alias("avg_pm10"),
            F.round(F.avg("co"),   2).alias("avg_co"),
            F.round(F.avg("no2"),  2).alias("avg_no2"),
            F.round(F.avg("so2"),  2).alias("avg_so2"),
            F.round(F.avg("o3"),   2).alias("avg_o3"),
        ).orderBy(F.desc("avg_pm25"))

        dangerous = clean_df.filter(
            F.col("air_quality_category").isin(
                "Unhealthy", "Very Unhealthy", "Hazardous"
            )
        ).groupBy("city", "air_quality_category").agg(
            F.count("*").alias("dangerous_count")
        ).orderBy(F.desc("dangerous_count"))

        # ── Save to SQL Server ────────────────────
        save_to_sql(clean_df,   "dbo.CleanedAirQuality",  if_exists="append")
        save_to_sql(avg_pm25,   "dbo.CitySummary",         if_exists="replace")
        save_to_sql(pollutants, "dbo.PollutantAvg",        if_exists="replace")
        save_to_sql(dangerous,  "dbo.DangerousAirQuality", if_exists="replace")

        print(f"  ✅ Batch {batch} done!")

    except Exception as e:
        import traceback
        print(f"  ❌ Batch {batch} failed:")
        traceback.print_exc()

    print(f"  Waiting {INTERVAL}s...")
    time.sleep(INTERVAL)

Batch loop started — Interrupt kernel to stop

[Batch 1] 2026-06-14 01:23:23
  Raw rows:   1423
  Clean rows: 1423

  Saving → dbo.CleanedAirQuality
    Rows: 1423
    ✅ Done → dbo.CleanedAirQuality

  Saving → dbo.CitySummary
    Rows: 5
    ✅ Done → dbo.CitySummary

  Saving → dbo.PollutantAvg
    Rows: 5
    ✅ Done → dbo.PollutantAvg

  Saving → dbo.DangerousAirQuality
    Rows: 0
    ✅ Done → dbo.DangerousAirQuality
  ✅ Batch 1 done!
  Waiting 60s...

[Batch 2] 2026-06-14 01:24:25
  Raw rows:   1433
  Clean rows: 1433

  Saving → dbo.CleanedAirQuality
    Rows: 1433
    ✅ Done → dbo.CleanedAirQuality

  Saving → dbo.CitySummary
    Rows: 5
    ✅ Done → dbo.CitySummary

  Saving → dbo.PollutantAvg
    Rows: 5
    ✅ Done → dbo.PollutantAvg

  Saving → dbo.DangerousAirQuality
    Rows: 0
    ✅ Done → dbo.DangerousAirQuality
  ✅ Batch 2 done!
  Waiting 60s...

[Batch 3] 2026-06-14 01:25:27
  Raw rows:   1438
  Clean rows: 1438

  Saving → dbo.CleanedAirQuality
    Rows: 1438
    ✅ Done